# Top+Side View Volume Estimation
# Frozen Feature Extractor + POV Fusion + Regression Head
# Leaving One Volume Out (LOVO)

This notebook trains and evaluates different combinations of frozen feature extractors, POV fusion strategies, and regression heads on the top+side dataset, always leaving one volume out (LOVO) and evaluating the performance on the held-out volume. There are 5 volume values in the dataset: 0, 1.06, 2.12, 3.18, and 4.24.


In [ ]:
import datetime
import warnings
from IPython.display import display
from sklearn.model_selection import ParameterGrid
from src.helpers import *
from src.frozen_pipeline import *
from src.training_and_evaluation import *
from src.constants import *

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

## Load Data


In [ ]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
print("Samples shape:", samples.shape)
display(samples[["exp_id", "volume", "top_path", "side_path"]].head(8))

## Show Regression Model Configurations


In [ ]:
print("Grid sizes per regressor:")
for name, (_, grid) in REGRESSION_MODEL_CONFIGS.items():
    n = len(list(ParameterGrid(grid))) if grid else 1
    print(f"  {name:15s}: {n:4d}")

## Nested CV Evaluation (split by volume values) - Backbone X Fusion Mode X Regression Model

In [ ]:
all_lovo_rows = []
all_lovo_results = {}
all_lovo_oof_predictions = {}
nested_artifacts = {}

for backbone_name in BACKBONE_NAMES:
    for fusion_name in FUSION_NAMES:
        print(f"\n{'#'*90}")
        print(f"LOVO | backbone={backbone_name} | fusion={fusion_name}")
        print(f"{'#'*90}")

        X, y, exp_groups = build_feature_matrix(
            samples=samples,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
            cache_dir=EMBEDDING_CACHE_DIR,
            device=DEVICE,
            top_mask_paths=TOP_ROI_MASKS,
            side_mask_paths=SIDE_ROI_MASKS
        )

        volume_groups = y.copy()

        lovo_results, lovo_oof_predictions = run_lovo_cv(
            X=X,
            y=y,
            exp_groups=exp_groups,         # inner CV grouped by experiment
            volume_groups=volume_groups,   # outer CV = leave one volume out
            model_configs=REGRESSION_MODEL_CONFIGS,
            inner_splits=5,
        )

        all_lovo_results[(backbone_name, fusion_name)] = lovo_results
        all_lovo_oof_predictions[(backbone_name, fusion_name)] = lovo_oof_predictions

        summary_df = summarise_nested_results(
            lovo_results,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
        )

        nested_artifacts[(backbone_name, fusion_name)] = {
            "X": X,
            "y": y,
            "groups": exp_groups,
            "nested_results": lovo_results,
            "oof_predictions": lovo_oof_predictions,
        }

        all_lovo_rows.append(summary_df)

all_lovo_summary = pd.concat(all_lovo_rows, ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean"]
).reset_index(drop=True)

## Add Dummy Mean Predictions for Comparison

In [ ]:
dummy_pred = np.full(len(samples), samples["volume"].mean(), dtype=float)
dummy_mae = mean_absolute_error(samples["volume"], dummy_pred)
dummy_mse = mean_squared_error(samples["volume"], dummy_pred)
dummy_rmse = np.sqrt(dummy_mse)
dummy_r2 = r2_score(samples["volume"], dummy_pred)

dummy_row = pd.DataFrame([{
    "backbone": "dummy_mean",
    "fusion": "dummy_mean",
    "regressor": "dummy_mean",
    "cv_mae_mean": dummy_mae,
    "cv_mae_std": 0.0,
    "cv_mse_mean": dummy_mse,
    "cv_mse_std": 0.0,
    "cv_rmse_mean": dummy_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": dummy_r2,
    "cv_r2_std": 0.0,
}])

results_df = pd.concat([all_lovo_summary, dummy_row], ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean", "cv_r2_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)

## Show Selected Hyperparameters

In [ ]:
for (backbone_name, fusion_name), artifact in nested_artifacts.items():
    print(f"\n{'='*90}")
    print(f"backbone={backbone_name}, fusion={fusion_name}")
    print(f"{'='*90}")
    for regressor_name, folds in artifact["nested_results"].items():
        has_params = any(f["best_params"] for f in folds)
        if not has_params:
            continue
        print(regressor_name)
        for f in folds:
            print(f"  Fold {f['fold']}: {f['best_params']}")
        print()

## Show Overall Performance for All Configurations (best at the top)
Prints mean and standard deviation of MAE, MSE, and RMSE over the CV folds. Plots out-of-fold predictions for the best configurations.

In [ ]:
display(all_lovo_summary)

for _, row in all_lovo_summary.head(5).iterrows():
    key = (row["backbone"], row["fusion"])
    artifact = nested_artifacts[key]
    y_true = artifact["y"]
    y_pred = artifact["oof_predictions"][row["regressor"]]
    title = f"{row['backbone']} | {row['fusion']} | {row['regressor']}"
    make_oof_plot(y_true, y_pred, title_prefix=title)

## Plot Predictions Per Withheld Volume

In [ ]:
lovo_prediction_rows = []

for (backbone_name, fusion_name), model_dict in all_lovo_results.items():
    X_tmp, y_tmp, _ = build_feature_matrix(
        samples=samples,
        backbone_name=backbone_name,
        fusion_name=fusion_name,
        cache_dir=EMBEDDING_CACHE_DIR,
        device=DEVICE,
        top_mask_paths=TOP_ROI_MASKS,
        side_mask_paths=SIDE_ROI_MASKS
    )

    for model_name, folds in model_dict.items():
        y_pred = all_lovo_oof_predictions[(backbone_name, fusion_name)][model_name]

        for i in range(len(y_tmp)):
            lovo_prediction_rows.append({
                "backbone": backbone_name,
                "fusion": fusion_name,
                "model": model_name,
                "true_volume": y_tmp[i],
                "pred_volume": y_pred[i],
                "held_out_volume": y_tmp[i],
                "config": f"{backbone_name} | {fusion_name} | {model_name}",
            })

leave_one_volume_predictions = pd.DataFrame(lovo_prediction_rows)

In [ ]:
unique_volumes = np.sort(leave_one_volume_predictions["held_out_volume"].unique())

display(all_lovo_summary)
PLOT_CONFIGS = all_lovo_summary.head(5).apply(
    lambda row: f"{row['backbone']} | {row['fusion']} | {row['regressor']}",
    axis=1
).tolist()

print("Predicted volumes on the held-out sets, for each selected config:")

for config_name in PLOT_CONFIGS:
    predictions = leave_one_volume_predictions[
        leave_one_volume_predictions["config"] == config_name
    ].copy()

    fig, axes = plt.subplots(
        1, len(unique_volumes),
        figsize=(4 * len(unique_volumes), 4),
        sharey=True,
    )
    if len(unique_volumes) == 1:
        axes = [axes]

    for ax, vol in zip(axes, unique_volumes):
        subset = predictions[predictions["held_out_volume"] == vol]
        counts, _, _ = ax.hist(subset["pred_volume"], bins=12, range=(0, 6))
        hist_height = max(80, int(np.max(counts) * 1.1))
        ax.set_xlim(0, 6)
        ax.set_ylim(0, hist_height)
        ax.axvline(vol, linestyle="--")
        ax.set_title(f"Held out {int(vol)}")
        ax.set_xlabel("Predicted volume")

    axes[0].set_ylabel(f"Count — {config_name}")
    plt.tight_layout()
    plt.show()

## Show Per Volume Performance for All Configurations
Sorted by the mean MAE on the highest volume value (76), since that seems to be most problematic to predict when left out.

In [ ]:
mae_per_volume_rows = []

for config_name, df_cfg in leave_one_volume_predictions.groupby("config"):
    backbone_name, fusion_name, model_name = config_name.split(" | ")

    for vol, df_vol in df_cfg.groupby("held_out_volume"):
        mae = np.mean(np.abs(df_vol["pred_volume"] - df_vol["true_volume"]))

        mae_per_volume_rows.append({
            "backbone": backbone_name,
            "fusion": fusion_name,
            "model": model_name,
            "held_out_volume": vol,
            "mae": mae,
        })

mae_per_volume_df = pd.DataFrame(mae_per_volume_rows)

mae_per_volume_wide = (
    mae_per_volume_df.pivot_table(
        index=["backbone", "fusion", "model"],
        columns="held_out_volume",
        values="mae",
        aggfunc="mean",
    )
    .reset_index()
)

mae_per_volume_wide.columns.name = None
mae_per_volume_wide = mae_per_volume_wide.rename(columns={
    0: "mae_mean_0",
    1.06: "mae_mean_1_06",
    2.12: "mae_mean_2_12",
    3.18: "mae_mean_3_18",
    4.24: "mae_mean_4_24",
})

display(mae_per_volume_wide.sort_values(["mae_mean_4_24"]))

## Save results

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
name = f"top_side_frozen_lovo_{timestamp}.csv"
all_lovo_summary.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)